In [1]:
from pathlib import Path
import argparse
import time

import torch
import random

from sklearn.metrics import mean_squared_error, mean_absolute_error

from utils import *
from model import *
from training_testing import *

import warnings
warnings.filterwarnings("ignore")

In [2]:
def get_arguments():
    parser = argparse.ArgumentParser(description="UDAR for Spatial-Temporal GNN in IoT Air Quality Multisensor Fusion", 
                                     add_help=False)

    # Data
    parser.add_argument("--data-dir", type=str, default="data/",
                        help='Path to the data')
    
    parser.add_argument("--source_dataset", type=str, default="R-212.csv",
                        help='Source dataset file')
    parser.add_argument("--target_dataset", type=str, default="R-69.csv",
                        help='Target dataset file')
    
    # Checkpoints
    parser.add_argument("--exp-dir", type=Path, default="exp/",
                        help='Path to the experiment folder, where all logs/checkpoints will be stored')

    # Optim
    parser.add_argument("--seed", type=int, default=42,
                        help='Seed for experiments')
    parser.add_argument("--epochs", type=int, default=150,
                        help='Number of epochs')
    parser.add_argument("--batch-size", type=int, default=64,
                        help='Batch size')
    parser.add_argument("--base-lr", type=float, default=3e-4,
                        help='Base Learning rate')

    # Running
    parser.add_argument("--num-workers", type=int, default=1)
    parser.add_argument('--device', default='cuda:1',
                        help='device to use for training / testing')
                        
    return parser

In [3]:
def seed_everything(seed = 0):   #42 #123 #0 #1 #1234
    r"""Sets the seed for generating random numbers in :pytorch:`PyTorch`,
    :obj:`numpy` and Python.

    Args:
        seed (int): The desired seed.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you are using multi-GPU.
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True

In [4]:
def mains(args):
    device = args.device if torch.cuda.is_available() else 'cpu'
    
    source_file = args.data_dir + args.source_dataset
    target_file = args.data_dir + args.target_dataset

    x_source, y_source, x_target, y_target, y_scaler = training_testing_dataframes(source_file,target_file)

    x_source, y_source, x_target, y_target = torch.from_numpy(x_source), torch.from_numpy(y_source),torch.from_numpy(x_target), torch.from_numpy(y_target)

    #Create Dataset
    data_x_y_source = torch.utils.data.TensorDataset(x_source, y_source)
    data_x_y_target = torch.utils.data.TensorDataset(x_target, y_target)

    #Create Dataloader
    train_dl_source = torch.utils.data.DataLoader(data_x_y_source, batch_size = args.batch_size, shuffle = False, drop_last = True, num_workers = args.num_workers)
    test_dl_source = torch.utils.data.DataLoader(data_x_y_source, batch_size= 1, shuffle = False)

    train_dl_target = torch.utils.data.DataLoader(data_x_y_target, batch_size = args.batch_size, shuffle = False, drop_last = True, num_workers = args.num_workers)
    test_dl_target = torch.utils.data.DataLoader(data_x_y_target, batch_size= 1, shuffle = False)
    
    #define Model
    model = FGNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.base_lr)
        
    model.train()

    for e in range(150):#args.epochs):
        t0 = time.time()
        train_loss = training_loop(model, optimizer, train_dl_source, train_dl_target, e, args.epochs, device, adaptation = 'fasttikuda')
        t1 = time.time()

        # print('Time of one iteration : ',t1-t0)
        
        # Print progress at every 5 epochs
        if (e + 1) % 5 == 0:
            print(f"Epoch: {e+1}")
            print('Training Loss:', train_loss)
        
    prediction, ground_truth = testing_loop(model,test_dl_target,device)

    print(f"NRMSE: {mean_squared_error(prediction,ground_truth, squared = False)}")
    print(f"NMAE: {mean_absolute_error(prediction,ground_truth)}")
        
        
    gt_denormalized = y_scaler.inverse_transform(ground_truth.reshape(-1, 1))
    pred_denormalized = y_scaler.inverse_transform(prediction.reshape(-1, 1))
    pred_denormalized = pred_denormalized.squeeze()


    print(f"RMSE: {mean_squared_error(pred_denormalized,gt_denormalized, squared = False)}")
    print(f"MAE: {mean_absolute_error(pred_denormalized,gt_denormalized)}")

In [5]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(parents=[get_arguments()])
    args = parser.parse_args([])
    
    seed_everything(args.seed)
    mains(args)

Epoch: 5
Training Loss: 0.03481003616682508
Epoch: 10
Training Loss: 0.03495661499486728
Epoch: 15
Training Loss: 0.03469793704535925
Epoch: 20
Training Loss: 0.03460313518051848
Epoch: 25
Training Loss: 0.03443421547611555
Epoch: 30
Training Loss: 0.034784781859453884
Epoch: 35
Training Loss: 0.034543166953054344
Epoch: 40
Training Loss: 0.03410996110037421
Epoch: 45
Training Loss: 0.03331420099306287
Epoch: 50
Training Loss: 0.029815728776156902
Epoch: 55
Training Loss: 0.009487621556741722
Epoch: 60
Training Loss: 0.008881148458881811
Epoch: 65
Training Loss: 0.008079835538272604
Epoch: 70
Training Loss: 0.007710804725348046
Epoch: 75
Training Loss: 0.007088386877016587
Epoch: 80
Training Loss: 0.006865303596538125
Epoch: 85
Training Loss: 0.006720141945124576
Epoch: 90
Training Loss: 0.0066899752286686135
Epoch: 95
Training Loss: 0.006366017994216897
Epoch: 100
Training Loss: 0.006178597764422496
Epoch: 105
Training Loss: 0.006030360878812093
Epoch: 110
Training Loss: 0.00607033229